In [0]:
%run ../setup/config

In [0]:
%run ../setup/utils

In [0]:
movies_metadata_df = spark.read.format("delta").load(f"{silver_folder_path}/movies_metadata")
ratings_df = spark.read.format("delta").load(f"{silver_folder_path}/ratings")
links_df = spark.read.format("delta").load(f"{silver_folder_path}/links")
cast_df = spark.read.format("delta").load(f"{silver_folder_path}/cast")

movies_metadata_df.printSchema()
ratings_df.printSchema()
links_df.printSchema()
cast_df.printSchema()


In [0]:
from pyspark.sql import functions as F

final_movies_df = (
    ratings_df.join(links_df, links_df.movie_id == ratings_df.movie_id, "inner")
    .join(movies_metadata_df, links_df.tmbd_id == movies_metadata_df.id, "inner")
    .join(cast_df, movies_metadata_df.id == cast_df.id, "inner")
    .groupBy(
        movies_metadata_df.id,
        "title",
        "cast_id",
        "cast_name",
        "cast_order",
        "collection_name",
    )
    .agg(
        F.avg("rating").alias("average_rating"),
        F.count("user_id").alias("number_of_ratings"),
    )
    .filter("cast_order == 0")
    .filter("collection_name = 'James Bond Collection'")
    .orderBy(F.col("average_rating").desc(), F.col("id"), F.col("cast_order").asc())
)
display(final_movies_df)

In [0]:
from pyspark.sql.window import Window

w = Window.partitionBy("cast_name")

df = (
    final_movies_df.withColumn("bond_rank", F.avg("average_rating").over(w))
    .orderBy(F.col("bond_rank").desc(), F.col("average_rating").desc())
    .drop(F.col("id"), F.col("cast_id"), F.col("cast_order"), F.col("number_of_ratings"), F.col("collection_name"))
)

df.display()

In [0]:
import plotly.express as px
import plotly.graph_objects as go

pdf = df.toPandas()
pdf["vs_actor_avg"] = pdf["average_rating"] - pdf["bond_rank"]

# --- 1) Films as points; actor average as a thick marker ---
fig = px.scatter(
  pdf,
  x="cast_name",
  y="average_rating",
  hover_name="title",
  color="cast_name",
  title="Bond films vs each actor's average rating",
  labels={
    "cast_name": "Bond actor",
    "average_rating": "Movie average rating",
  },
  range_y=[3.1, 3.9],
)

actor_avg = pdf[["cast_name", "bond_rank"]].drop_duplicates()
fig.add_trace(
  go.Scatter(
    x=actor_avg["cast_name"],
    y=actor_avg["bond_rank"],
    mode="markers",
    marker=dict(symbol="line-ew", size=28, line=dict(width=4, color="white")),
    name="Actor avg (window)",
    hovertemplate="Actor avg: %{y:.3f}<extra></extra>",
  )
)
fig.update_layout(showlegend=True, height=500)
fig.show()